# Mapping Hotspot Gene Modules to Malignant Cell Transcriptional States

Maps Hotspot-derived gene modules to literature-informed transcriptional
states, assigns each malignant cell a dominant (and possible hybrid) state
using a percentile-based threshold, summarizes state composition across
patients/RECIST/smoking/EGFR groups, and tests state-gene expression and
state-vs-response associations.

In [ ]:
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/tumour/pulmonary_epi_Lin/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
from scipy.stats import zscore, fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests

## Note: Change 'Neurod_EMT-like' to 'Neural_EMT-like' in the text

## Config

In [ ]:
cluster_input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06.h5ad"
hotspot_scored_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_hvg2000.h5ad"
output_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states.h5ad"
figures_output_dir = "./figures"
enrichment_output_dir = "./enrichment_results"

malignant_label = "Malignant"

# maps each Hotspot module to a literature-informed transcriptional state
module_map = {
    "Module 13": "1_Epithelial baseline",  # cancer cell located in adjacent normal
    "Module 10": "1_Epithelial baseline",
    "Module 11": "1_Epithelial baseline",
    "Module 15": "1_Epithelial baseline",
    "Module 12": "1_Epithelial baseline",
    "Module 7": "1_Epithelial baseline",
    "Module 4": "3_Devp_stem-like",
    "Module 6": "3_Devp_stem-like",
    "Module 5": "2_Differentiated epithelial",
    "Module 14": "4_Inflammatory EMT",  # stress
    "Module 9": "4_Inflammatory EMT",   # stress
    "Module 8": "5_Neurod_EMT-like",
    "Module 1": "6_Proliferative",
    "Module 3": "5_Neurod_EMT-like",
    "Module 2": "5_Neurod_EMT-like",
}

# percentile threshold used to call a state "active" in a cell, plus the
# sensitivity range explored before settling on CHOSEN_PCT
sensitivity_percentiles = np.arange(0.60, 0.95, 0.05)
CHOSEN_PCT = 0.75
state_col = f"dominant_state_p{int(CHOSEN_PCT * 100)}"
states_col = f"states_p{int(CHOSEN_PCT * 100)}"

state_order = [
    "1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
    "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative",
]
state_colors = {
    "1_Epithelial baseline": "#8dd3c7",
    "2_Differentiated epithelial": "#bebada",
    "3_Devp_stem-like": "#80b1d3",
    "4_Inflammatory EMT": "#fccde5",
    "5_Neurod_EMT-like": "#bc80bd",
    "6_Proliferative": "#ffed6f",
    "unassigned": "grey",
    "None": "grey",
}

treatment_order = ["1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
recist_order = ["0_adj_normal", "1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
smoking_order = ["non_smoker", "smoker", "unknown"]
egfr_order = ["L858R", "exon19del", "L858R_T790M"]

# patient display order for composition barplots (adjacent-normal samples first)
donor_order = [
    "TN009_AN", "TN006_AN", "TN002_AN", "TN003_AN", "TN007_AN", "TN005_AN", "TN004_AN", "TN001_AN",
    "TN007", "TN005", "TN008", "TN006", "TN002", "TN001", "TN009", "TN004", "TN003",
    "P012", "P009", "P011", "P003", "P001", "P005", "P006", "P008", "P002", "P013",
    "R006", "R009", "R005", "R010", "R007", "R003", "R001",
]

# minimum cell count for a hybrid state combination to be shown
min_hybrid_state_cells = 50

# --- data-cleaning maps (metadata label fixes; see "Data cleaning" section) ---
egfr_relabel_map = {"exon19": "exon19del"}
p53_truncating_categories = ["nonsense", "frameshift_indel", "nonsense_frameshift_indel"]

# --- marker gene panels for dotplot visualization ---
marker_panels = {
    "hotspot_markers": {
        "1": {"CENPP", "MKI67"},
        "2": {"NLGN1", "SEMA3A", "SOX4", "NRCAM", "LGR4", "TP63", "CDH11"},
        "3": {"ITGB3", "CXCL14", "IFI44L"},
        "4": {"MUC4", "MUC21", "CEACAM5"},
        "5": {"LGR5", "PAX7", "SOX5", "SOX6", "ETV5", "PBX1", "ROR1", "ROR2"},
        "6": {"KRT7", "AREG", "TGFB2", "FGF13", "ITGA1", "ERBB4", "SFTPB", "CD55"},
    },
    "ADC_targets": {
        "1": {"TACSTD2", "ERBB2", "EGFR", "MET", "ERBB3"},
        "2": {"CD276", "NECTIN4", "PTK2", "PTK7", "SLC39A6"},
        "3": {"MSLN", "DLL3", "SLC3A2", "ITGA2", "ITGA3", "EPHA2", "TFRC"},
        "4": {"CD47", "CEACAM5", "CEACAM6"},
    },
    "neuroendocrine": {
        "NE TF": {"ASCL1", "NEUROD1", "INSM1", "POU2F3"},
        "secretory": {"SYP", "CHGA", "CGA", "SCG2"},
        "NE synaptic adhesion": {"SYN3", "SYT6", "NLGN1", "GRIA4", "SLC6A3", "CNTNAP2", "NGF", "CDH11"},
    },
}

# Fisher exact test config: compares one state's prevalence in target_group vs each comparison_group
fisher_target_state = "4_Inflammatory EMT"
fisher_target_group = "3_on_treatment"
fisher_comparison_groups = ["0_adj_normal", "1_treatment_naive", "4_TKI_resistance"]

import os
os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(enrichment_output_dir, exist_ok=True)

## 1. Load data and merge Hotspot module scores

In [ ]:
adata = sc.read(cluster_input_path)
adata = adata[adata.obs["cell_type_new"] == malignant_label].copy()

# transfer module-score columns computed on the HVG-scored malignant subset
adata_scored = sc.read(hotspot_scored_path)
module_cols = [col for col in adata_scored.obs.columns if col.startswith("Module ")]
assert all(adata.obs_names == adata_scored.obs_names), "cell order/identity mismatch between the two files"
adata.obs[module_cols] = adata_scored.obs[module_cols]
del adata_scored

### Sanity check: are module scores raw or Z-scored?

Inspect the score distribution directly from `adata.obs` before deciding whether to Z-score.

In [ ]:
adata.obs[module_cols].agg(["mean", "std", "min", "max"]).T

## 2. Map modules to states and reshape to long format

In [ ]:
df = adata.obs.copy()
df[module_cols] = df[module_cols].apply(zscore)  # keeps scores comparable across modules for tie-breaking

long_df = df[["ID_2", "recist_v3", "smoking_status_grp"] + module_cols].copy()
long_df["cell"] = long_df.index
long_df = long_df.melt(
    id_vars=["ID_2", "recist_v3", "cell", "smoking_status_grp"],
    value_vars=module_cols, var_name="module", value_name="score",
)
long_df["state"] = long_df["module"].map(module_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
long_df["score"].hist(bins=100, ax=axes[0])
axes[0].set_title("Score distribution (all cells)")
axes[0].set_xlabel("Z-score")
for mod, grp in long_df.groupby("module"):
    axes[1].hist(grp["score"], bins=50, alpha=0.3, label=mod)
axes[1].set_title("Per-module score distributions")
axes[1].set_xlabel("Z-score")
plt.tight_layout()
plt.show()

## 3. Percentile threshold sensitivity analysis

For each candidate percentile, computes the per-module score threshold and the resulting fraction of cells with zero, one, or multiple ("hybrid") active states.

In [ ]:
results = []
for pct in sensitivity_percentiles:
    thresh_map = long_df.groupby("module")["score"].quantile(pct)
    filtered = long_df[long_df["score"] > long_df["module"].map(thresh_map)]

    n_programs = (
        filtered.groupby("cell")["state"].nunique().reindex(adata.obs.index).fillna(0)
    )
    total_cells = len(n_programs)
    results.append({
        "percentile": pct,
        "none": (n_programs == 0).sum() / total_cells,
        "single": (n_programs == 1).sum() / total_cells,
        "hybrid": (n_programs >= 2).sum() / total_cells,
        "threshold_mean": thresh_map.mean(),
    })

plasticity_df = pd.DataFrame(results)
print(plasticity_df)

plt.figure(figsize=(5, 4))
plt.plot(plasticity_df["percentile"], plasticity_df["hybrid"], marker="o", linewidth=2, label="Hybrid (\u22652 programs)")
plt.plot(plasticity_df["percentile"], plasticity_df["single"], marker="o", linewidth=2, label="Single program")
plt.plot(plasticity_df["percentile"], plasticity_df["none"], marker="o", linewidth=2, label="None")
plt.xlabel("Percentile threshold")
plt.ylabel("Fraction of cells")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Assign states at the chosen threshold

In [ ]:
thresh_map = long_df.groupby("module")["score"].quantile(CHOSEN_PCT)
long_df_filtered = long_df[long_df["score"] > long_df["module"].map(thresh_map)]

# all active states per cell ("hybrid" label if more than one)
state_series = long_df_filtered.groupby("cell")["state"].apply(lambda x: "|".join(sorted(set(x))))
adata.obs[states_col] = adata.obs.index.map(state_series).fillna("None").astype("category")

# dominant state: highest-scoring active module wins
dominant_state = (
    long_df_filtered.sort_values("score", ascending=False)
    .drop_duplicates(subset="cell")
    .set_index("cell")["state"]
)
adata.obs[state_col] = adata.obs.index.map(dominant_state).fillna("None").astype("category")

adata.obs[state_col].value_counts()

## 5. State composition across groups

One reusable function replaces what were six separate, nearly-identical stacked-barplot cells in the original notebook (composition by patient, by RECIST, by smoking status, and by EGFR mutation, for both the hybrid-state and dominant-state columns). All of these can be computed as a single groupby directly on `adata.obs` — the original's per-patient aggregate-then-remap approach gives an identical result, just via an unnecessary extra step.

In [ ]:
def plot_state_composition(obs_df, state_col, groupby_col, group_order=None, colors=None,
                            title="", xlabel=None, figsize=(5, 4), save_path=None):
    """Stacked barplot of state_col composition (fraction) within each groupby_col group."""
    count_df = obs_df.groupby([groupby_col, state_col], observed=False).size().reset_index(name="count")
    count_df["fraction"] = count_df.groupby(groupby_col, observed=False)["count"].transform(lambda x: x / x.sum())
    plot_df_wide = count_df.pivot(index=groupby_col, columns=state_col, values="fraction").fillna(0)
    if group_order:
        plot_df_wide = plot_df_wide.reindex(group_order)
    colors_to_use = [colors.get(c, "grey") for c in plot_df_wide.columns] if colors else None

    ax = plot_df_wide.plot(kind="bar", stacked=True, figsize=figsize, color=colors_to_use, width=0.7)
    ax.set_ylabel("Proportion of cells")
    ax.set_xlabel(xlabel or groupby_col)
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title=state_col)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return plot_df_wide

In [ ]:
plot_state_composition(
    adata.obs, state_col=state_col, groupby_col="ID_2", group_order=donor_order, colors=state_colors,
    title=f"Cancer Cell States per Patient (p{int(CHOSEN_PCT*100)})",
    save_path=f"{figures_output_dir}/1_barplot_patient_cancer_cell_state_p{int(CHOSEN_PCT*100)}.pdf",
)
plot_state_composition(
    adata.obs, state_col=state_col, groupby_col="recist_v3", group_order=recist_order, colors=state_colors,
    title=f"Cancer Cell State Distribution by RECIST Group (p{int(CHOSEN_PCT*100)})",
    save_path=f"{figures_output_dir}/2_barplot_cancer_cell_state_by_RECIST_p{int(CHOSEN_PCT*100)}.pdf", #Figure 2 
)
plot_state_composition(
    adata.obs, state_col=state_col, groupby_col="smoking_status_grp", group_order=smoking_order, colors=state_colors,
    title="Cancer Cell State Distribution by Smoking Group",
    save_path=f"{figures_output_dir}/3_barplot_cancer_cell_state_by_smoking_p{int(CHOSEN_PCT*100)}.pdf",
)
plot_state_composition(
    adata.obs, state_col=state_col, groupby_col="EGFR_mutation", group_order=egfr_order, colors=state_colors,
    title="Cancer Cell State Distribution by EGFR mutation", figsize=(2, 4),
    save_path=f"{figures_output_dir}/3_barplot_cancer_cell_state_by_EGFRm.pdf",
)

## 6. Hybrid state distribution by RECIST group

In [ ]:
colors_30 = sns.color_palette("tab20", 20) + sns.color_palette("tab20b", 35)

df_hybrid = adata.obs[adata.obs[states_col] != "None"]
hybrid_counts = df_hybrid.groupby(["recist_v3", states_col], observed=False).size().reset_index(name="count")
hybrid_counts["fraction"] = hybrid_counts.groupby("recist_v3", observed=False)["count"].transform(lambda x: x / x.sum())

top_states = df_hybrid[states_col].value_counts()
top_states = top_states[top_states > min_hybrid_state_cells].index
hybrid_counts = hybrid_counts[hybrid_counts[states_col].isin(top_states)]

plot_df = hybrid_counts.pivot(index="recist_v3", columns=states_col, values="fraction").fillna(0).reindex(recist_order)

plot_df.plot(kind="bar", stacked=True, figsize=(4, 8), color=colors_30[: len(plot_df.columns)])
plt.ylabel("Fraction of cells")
plt.xlabel("RECIST group")
plt.title("Hybrid cancer cell state distribution by RECIST group")
plt.legend(bbox_to_anchor=(1.05, 1), title="Hybrid states")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/4_hybrid_states_by_recist_top{min_hybrid_state_cells}.pdf", dpi=300)
plt.show()

plot_df_reset = plot_df.reset_index().melt(id_vars="recist_v3", var_name="state", value_name="fraction")
plt.figure(figsize=(6, 5))
sns.scatterplot(
    data=plot_df_reset, x="recist_v3", y="state", size="fraction", hue="fraction",
    palette="viridis", sizes=(20, 200), legend="brief",
)
plt.xticks(rotation=45)
plt.ylabel("Hybrid state")
plt.xlabel("RECIST group")
plt.title("Hybrid cell state distribution")
plt.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

## 7. Visualize states on UMAP / FDG

In [ ]:
sc.pl.umap(adata, color=states_col, palette="tab20")
sc.pl.umap(adata, color=state_col, palette="Set2")
sc.pl.draw_graph(adata, color=[state_col], size=30)

## 8. Save state-annotated output

In [ ]:
adata.X = adata.layers["raw_counts"].copy()
adata.write(output_path)
adata

## 9. Data cleaning: metadata label fixes

Fixes inconsistent EGFR mutation labels and collapses TP53 mutation subtypes into broader categories. Run this after reloading a previously saved state-annotated object if picking the analysis back up later.

In [ ]:
adata.obs["EGFR_mutation"] = (
    adata.obs["EGFR_mutation"].map(lambda x: egfr_relabel_map.get(x, x)).astype("category")
)

if "unknown_AN" not in adata.obs["p53_mutation"].cat.categories:
    adata.obs["p53_mutation"] = adata.obs["p53_mutation"].cat.add_categories(["unknown_AN"])
adata.obs.loc[adata.obs["ID_2"].str.endswith("_AN"), "p53_mutation"] = "unknown_AN"

if "truncating" not in adata.obs["p53_mutation"].cat.categories:
    adata.obs["p53_mutation"] = adata.obs["p53_mutation"].cat.add_categories(["truncating"])
adata.obs.loc[adata.obs["p53_mutation"].isin(p53_truncating_categories), "p53_mutation"] = "truncating"
adata.obs["p53_mutation"] = adata.obs["p53_mutation"].cat.remove_unused_categories()

# drop unassigned cells for subsequent analyses
adata = adata[adata.obs[state_col] != "None"].copy()

## 10. Gene expression by state and RECIST group

In [ ]:
adata.layers["raw_counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

for panel_name, genes in marker_panels.items():
    sc.pl.dotplot(
        adata, genes, groupby="recist_v3", use_raw=False, standard_scale="var", cmap="Blues",
        figsize=(10, 3.5), dendrogram=False, save=f"_{panel_name}_by_recist.pdf",
    )
    sc.pl.dotplot(
        adata, genes, groupby=state_col, use_raw=False, standard_scale="var", cmap="Blues",
        figsize=(4.2, 7), swap_axes=True, dendrogram=False, save=f"_{panel_name}_by_state.pdf", #Figure 2E
    )

## 11. Combined RECIST x state comparison

In [ ]:
adata.obs["recist_state"] = adata.obs["recist_v3"].astype(str) + " | " + adata.obs[state_col].astype(str)
adata_plot = adata[~adata.obs["recist_state"].str.contains("0_adj_normal", na=False)].copy()

desired_order = [
    f"{treatment} | {state}"
    for treatment in treatment_order
    for state in state_order
    if f"{treatment} | {state}" in adata_plot.obs["recist_state"].unique()
]
adata_plot.obs["recist_state"] = pd.Categorical(adata_plot.obs["recist_state"], categories=desired_order, ordered=True)

sc.pl.dotplot(
    adata_plot, marker_panels["ADC_targets"], groupby="recist_state", use_raw=False,
    standard_scale="var", cmap="Blues", figsize=(8, 9), swap_axes=True, dendrogram=False,
    save="_ADC_targets_by_recist_state.pdf", #Figure 3I
)

## 12. Differential expression per state, across RECIST

In [ ]:
for state in state_order:
    print(f"Processing state: {state}")
    adata_state = adata[adata.obs[state_col] == state].copy()

    # skip RECIST groups with too few cells for a valid wilcoxon comparison
    group_sizes = adata_state.obs["recist_v3"].value_counts()
    valid_groups = group_sizes[group_sizes >= 2].index.tolist()
    if len(valid_groups) < 2:
        print(f"  Skipping {state}: fewer than 2 RECIST groups have >=2 cells")
        continue
    adata_state = adata_state[adata_state.obs["recist_v3"].isin(valid_groups)].copy()

    sc.tl.rank_genes_groups(adata_state, groupby="recist_v3", method="wilcoxon", pts=True)
    sc.pl.rank_genes_groups(adata_state, n_genes=30, sharey=False, title=f"Top DE genes for {state}")